# Unidad 2: Aprendizaje Automático 
## 🌲 Random Forest Classifier
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER


[Foto de Janis Zv](https://www.pexels.com/es-es/foto/verano-bosque-arboles-medio-ambiente-18486931/)

---

## 🎯 ¿Qué vamos a aprender?

En este notebook vamos a explorar **Random Forest**, uno de los algoritmos de Machine Learning más potentes y versátiles.

Al finalizar, vas a poder:
- ✅ Entender qué es un **Bosque Aleatorio** y cómo difiere de un único Árbol de Decisión
- ✅ Comprender el concepto de **Ensemble Learning** (aprendizaje por conjuntos)
- ✅ Entrenar y comparar **RandomForest**, **Logistic Regression** y **DecisionTree** sobre el mismo dataset
- ✅ Evaluar métricas: Accuracy, Precision, Recall y F1
- ✅ Inspeccionar los hiperparámetros por defecto del modelo

---

## 🌲 ¿Qué es Random Forest?

**Random Forest** es un algoritmo de **aprendizaje supervisado** que pertenece al grupo de los **clasificadores de ensamble** (*Ensemble Learning*). En lugar de entrenar un solo modelo, entrena **muchos árboles de decisión** y combina sus predicciones.

### 🧩 Idea central: el "poder de la multitud"

Imaginá que le preguntás a **1 experto** vs **100 expertos independientes** sobre un diagnóstico. La votación de 100 expertos suele ser más acertada que la opinión de uno solo. Eso es exactamente lo que hace Random Forest.

```
         Dataset
        /   |   \
  Árbol1  Árbol2  Árbol3  ...  Árbol_n
      \    |    /
       Votación
          |
     Predicción final
```

### 🎲 Fuentes de aleatoriedad (¿por qué se llama "Random"?)

Cada árbol se entrena con:
1. **Bootstrap Sampling**: un subconjunto aleatorio *con reemplazo* de las muestras de entrenamiento.
2. **Feature Randomness**: en cada división, sólo se evalúa un subconjunto aleatorio de las variables de entrada.

Esta doble aleatoriedad hace que los árboles sean **independientes** entre sí, reduciendo la varianza y evitando el overfitting.

### 🆚 Comparación con Árbol de Decisión simple

| Característica | Árbol de Decisión | Random Forest |
|----------------|------------------|---------------|
| Número de modelos | 1 | Muchos (n_estimators) |
| Overfitting | Alto riesgo | Bajo riesgo |
| Interpretabilidad | Alta ✅ | Baja ⚠️ |
| Rendimiento | Bueno | Muy bueno ✅ |
| Tiempo de entrenamiento | Rápido | Más lento |

---

## 🩺 El Dataset: Breast Cancer Wisconsin

Utilizaremos el clásico dataset **Breast Cancer Wisconsin** incluido en scikit-learn.

- **569** muestras de tumores
- **30** características numéricas (radio, textura, perímetro, área, etc.)
- **2 clases**: `0` = Maligno, `1` = Benigno

El objetivo es clasificar tumores como **malignos** o **benignos** a partir de mediciones del núcleo celular.

> 📌 **Referencias:**
> - Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
> - Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1994). [Machine learning techniques to diagnose breast cancer...](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC200909/). *Cancer Letters*, 77(2-3), 163–171.
> - Scikit-learn: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

---

## 📦 Paso 1: Importar las Librerías

Importamos las herramientas necesarias:

- 🐼 **pandas**: manipulación de datos
- 🌲 **RandomForestClassifier**: el algoritmo principal
- 📈 **LogisticRegression**: modelo de referencia
- 🌳 **DecisionTreeClassifier**: modelo de referencia
- ✂️ **train_test_split**: división train/test
- 📊 **metrics**: métricas de evaluación

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# 🌲 Clasificadores
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# ✂️ División de datos y métricas
from sklearn.model_selection import train_test_split
from sklearn import metrics

pd.set_option('display.max_columns', None)
print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar y Explorar el Dataset

Cargamos el dataset **Breast Cancer Wisconsin** directamente desde scikit-learn. Es uno de los datasets clásicos para practicar clasificación binaria.

> 💡 **Tip:** Los datasets incluidos en scikit-learn son ideales para aprender y experimentar, ya que ya están limpios y listos para usar.

In [ ]:
# 📥 Cargar el dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data['data'], columns=cancer_data['feature_names'])
df['target'] = cancer_data['target']

print(f'📐 Dimensiones: {df.shape[0]} muestras x {df.shape[1]} columnas')
print(f'🏷️  Clases: {list(cancer_data.target_names)}')
print(f'\n📊 Distribución de clases:')
print(df['target'].value_counts().rename({0: 'Maligno (0)', 1: 'Benigno (1)'}))
df.head()

In [ ]:
# 🔢 Estadísticas descriptivas
print('📈 Estadísticas descriptivas (primeras 10 columnas):')
df[cancer_data.feature_names[:10]].describe()

## 🎯 Paso 3: Preparar los Datos

Separamos las **features (X)** del **target (y)** y dividimos en conjuntos de **entrenamiento** y **prueba**.

Usamos `random_state=144` para garantizar reproducibilidad.

In [ ]:
# 🎯 Features y Target
X = df[cancer_data.feature_names].values
y = df['target'].values

print(f'📐 Dimensiones de X: {X.shape}')

# ✂️ División entrenamiento / prueba (75% / 25% por defecto)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=144)

print(f'🏋️  Entrenamiento: {X_train.shape[0]} muestras')
print(f'🧪  Prueba:        {X_test.shape[0]} muestras')

## 🤖 Paso 4: Entrenar los Tres Modelos

Vamos a entrenar tres clasificadores con exactamente los **mismos datos** para comparar su rendimiento:

| Modelo | Tipo | Descripción |
|--------|------|-------------|
| 🌲 **Random Forest** | Ensemble | Muchos árboles que votan |
| 🌳 **Decision Tree** | Instancia única | Un solo árbol |
| 📈 **Logistic Regression** | Lineal | Probabilidad logística |

In [ ]:
# 🌲 Random Forest (con parámetros por defecto)
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# 📈 Logistic Regression
lr = LogisticRegression(solver='liblinear')
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# 🌳 Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('✅ Los tres modelos fueron entrenados correctamente!')
print('\n🔧 Hiperparámetros por defecto de Random Forest:')
print(rf.get_params())

## 📊 Paso 5: Comparar Métricas

Evaluamos los tres modelos usando las métricas más importantes para clasificación binaria:

| Métrica | ¿Qué mide? |
|---------|------------|
| **Accuracy** | % de predicciones correctas en total |
| **Precision** | De los que predije positivos, ¿cuántos eran realmente positivos? |
| **Recall** | De todos los positivos reales, ¿cuántos detecté? |
| **F1** | Media armónica entre Precision y Recall |

> 🔑 **¿Cuándo usar cada métrica?**
> - Si los errores de **falsos negativos** son costosos → priorizar **Recall** (ej: detección de cáncer)
> - Si los errores de **falsos positivos** son costosos → priorizar **Precision** (ej: spam filter)
> - Para un balance general → usar **F1**

In [ ]:
def evaluar_modelo(nombre, modelo, X_te, y_te, y_pred):
    print(f'\n{nombre}')
    print(f'     Score:    {modelo.score(X_te, y_te):.4f}')
    print(f'  Accuracy:    {metrics.accuracy_score(y_te, y_pred):.4f}')
    print(f'  Precision:   {metrics.precision_score(y_te, y_pred):.4f}')
    print(f'  Recall:      {metrics.recall_score(y_te, y_pred):.4f}')
    print(f'  F1:          {metrics.f1_score(y_te, y_pred):.4f}')

print('=' * 50)
print('        📊 COMPARACIÓN DE MODELOS')
print('=' * 50)
evaluar_modelo('📈 Logistic Regression:', lr, X_test, y_test, y_pred_lr)
evaluar_modelo('🌲 Random Forest:',       rf, X_test, y_test, y_pred_rf)
evaluar_modelo('🌳 Decision Tree:',       dt, X_test, y_test, y_pred_dt)
print('=' * 50)

## 📈 Paso 6: Visualización Comparativa

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos para graficar
modelos = ['Logistic\nRegression', 'Random\nForest', 'Decision\nTree']
colores = ['steelblue', 'forestgreen', 'darkorange']

def get_metrics(modelo, X_te, y_te):
    y_p = modelo.predict(X_te)
    return [
        metrics.accuracy_score(y_te, y_p),
        metrics.precision_score(y_te, y_p),
        metrics.recall_score(y_te, y_p),
        metrics.f1_score(y_te, y_p)
    ]

vals_lr = get_metrics(lr, X_test, y_test)
vals_rf = get_metrics(rf, X_test, y_test)
vals_dt = get_metrics(dt, X_test, y_test)

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, vals_lr, width, label='Logistic Regression', color='steelblue', edgecolor='black')
ax.bar(x,         vals_rf, width, label='Random Forest',       color='forestgreen', edgecolor='black')
ax.bar(x + width, vals_dt, width, label='Decision Tree',       color='darkorange', edgecolor='black')

ax.set_ylabel('Score')
ax.set_title('Comparación de Métricas entre Modelos', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0.85, 1.02)
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'{val:.2f}'))
plt.tight_layout()
plt.show()

print('\n💡 Observación: Random Forest generalmente supera a un único árbol de decisión')
print('   y es competitivo con Logistic Regression, con mayor robustez.')

## 🏁 Conclusiones

En este notebook aprendimos:

1. 🌲 **Random Forest** combina múltiples Árboles de Decisión para reducir el overfitting y mejorar la generalización.
2. 🎲 La aleatoriedad (bootstrap + feature sampling) hace que los árboles sean **independientes** entre sí, lo que reduce la varianza del modelo.
3. 📊 Random Forest generalmente supera a un solo árbol de decisión en métricas como Accuracy, Precision y Recall.
4. 🔧 Los parámetros por defecto de scikit-learn ya dan buenos resultados, pero se pueden **tunear** para mejorar aún más el rendimiento.

### ➡️ Próximo notebook: Tuning de Hiperparámetros con GridSearchCV

---

## 📚 Referencias

- Breiman, L. (2001). [Random Forests](https://link.springer.com/article/10.1023/A:1010933404324). *Machine Learning*, 45, 5–32.
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd ed.). O'Reilly Media.
- Scikit-learn Documentation: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- Scikit-learn Documentation: [Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)